# dfs-three-set-toposort — worked example 2: Topological sort of a linear chain of five nodes

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `dfs-three-set-toposort`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """A minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries an optional `.recipe`
    populated by wrap_forward_fn. `requires_grad` is set by the wrapper.
    `.grad` accumulates the leaf gradient at the end of the reverse pass."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
        self.grad = None
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

When the graph is a simple chain (no branching), the DFS toposort still applies the same three-set logic. Each node is entered, its single child is visited recursively to completion, then the current node is appended. The result is the chain in reverse insertion order: deepest leaf first, root last.

## Worked solution

Chain: n0 → n1 → n2 → n3 → n4 (n4 has no children).

`visit(n0)`: add n0 to temp, recurse into n1.
`visit(n1)`: add n1 to temp, recurse into n2.
`visit(n2)`: add n2 to temp, recurse into n3.
`visit(n3)`: add n3 to temp, recurse into n4.
`visit(n4)`: add n4 to temp, no children. Move n4 to perm, append. Result: `[n4]`.

Unwinding: n3 finishes → appended. n2 → appended. n1 → appended. n0 → appended.

Final result: `[n4, n3, n2, n1, n0]`. The deepest node (`n4`, the leaf / pure input) comes first; the root (`n0`, the output) is last. This is the correct deps-first order: each node appears after everything it depends on.

In [ ]:
def topological_sort_chain(n):
    """Sort a linear chain 0 -> 1 -> 2 -> ... -> n-1."""
    # Represent nodes as integers; children[i] = [i+1] except for the last.
    children = {i: [i + 1] for i in range(n - 1)}
    children[n - 1] = []

    result = []
    perm = set()
    temp = set()

    def visit(node):
        if node in perm:
            return
        if node in temp:
            raise ValueError(f'Cycle at {node}')
        temp.add(node)
        for child in children[node]:
            visit(child)
        temp.discard(node)
        perm.add(node)
        result.append(node)

    visit(0)
    return result

ordering = topological_sort_chain(5)
print('Chain order (deps first):', ordering)
print('Leaf 4 is first:', ordering[0] == 4)
print('Root 0 is last:', ordering[-1] == 0)
print('Every node present exactly once:', sorted(ordering) == list(range(5)))